In [ ]:
config_path = "config/car_coll/v1_actuarial/config.yaml"

In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

# Add lib folder to path for encoding_strategies
if not os.path.exists('encoding_strategies.py'):
    sys.path.insert(0, '../lib')
from encoding_strategies import encode_type3_actuarial

# Detect if running self-contained (from output folder) or via papermill (from project root)
# Self-contained mode: local files exist in same directory as notebook
SELF_CONTAINED = os.path.exists('gbm_functions.ipynb') and os.path.exists('config.yaml')

if SELF_CONTAINED:
    # Running from output folder - use local files
    sys.path.insert(0, '.')  # Add current dir for utils import
    from utils import load_config
    config_path = 'config.yaml'
    config_dir = '.'
    pc_file = '../../../.PC'  # Navigate up to project root for .PC
    print("Mode: Self-contained (running from output folder)")
else:
    # Running via papermill from project root
    from utils import load_config
    config_dir = os.path.dirname(config_path)
    pc_file = '.PC'
    print("Mode: Papermill (running from project root)")

# Load config with machine-specific paths
cfg = load_config(config_path)

print(f"Experiment: {cfg['experiment']['name']}")
print(f"Machine: PC{open(pc_file).read().strip()}")
print(f"Config: {config_path}")
print(f"Timestamp: {datetime.now()}")

In [ ]:
# Load shared functions - path depends on run mode
if SELF_CONTAINED:
    %run ./gbm_functions.ipynb
else:
    %run ./edited_20240314_symbols_gbm_functions.ipynb

In [ ]:
# Extract config values
veh_type = cfg['experiment']['veh_type']
cov = cfg['experiment']['coverage']

# Machine-specific paths
paths = cfg['machine']['paths']
path_prefix = paths['path_prefix']
in_data_store = path_prefix + paths['input_data_dir']
out_data_store = path_prefix + 'dataout/GBMFirstOutdata/'
ep_path = path_prefix + paths['ep_data_dir']
support_path = path_prefix + paths['support_dir']

DEBUG = cfg['model']['debug']
DEBUG_ROWS = cfg['model']['debug_rows']

print(f"Vehicle Type: {veh_type}")
print(f"Coverage: {cov}")
print(f"Path Prefix: {path_prefix}")
print(f"Debug Mode: {DEBUG}")

In [ ]:
# Type 3 Actuarial Encoding - uses level_mapping.csv instead of feature_selection.csv
print("Using Type 3 Actuarial Encoding")
# The encoding will be applied after data loading
# level_mapping.csv path will be handled by encoding_strategies module

In [ ]:
# Load feature clipping (optional)
clipping_path = os.path.join(config_dir, cfg['files'].get('feature_clipping', 'feature_clipping.csv'))
if os.path.exists(clipping_path):
    clipping_df = pd.read_csv(clipping_path)
    print(f"Clipping rules loaded: {len(clipping_df)}")
    display(clipping_df)
else:
    clipping_df = None
    print("No clipping rules")

In [ ]:
# Load EP data
ep_file = cfg['files']['ep_file']
data_ep = pd.read_csv(ep_path + ep_file)
print(f"EP data shape: {data_ep.shape}")

In [ ]:
# Load main data
input_file = cfg['files']['input_file_pattern'].format(veh_type=veh_type)
input_path = in_data_store + input_file

if DEBUG:
    data_ini = pd.read_csv(input_path, nrows=DEBUG_ROWS)
else:
    data_ini = pd.read_csv(input_path)

print(f"Data shape: {data_ini.shape}")
display(data_ini.head())

In [ ]:
# Compute derived columns
data_ini['offset_AME_VT_VV_coll'] = data_ini['veh_type_factor_coll'] * data_ini['AME_offset_coll'] * data_ini['coll_primary_VV_offset']
data_ini['incurred_act_capped'] = data_ini['incurred_cap_AME_VV_VT_coll'] * data_ini['offset_AME_VT_VV_coll']

In [ ]:
data = data_ini.copy()

data['offset_AME_VT_VV_coll'] = data['veh_type_factor_coll'] * data['AME_offset_coll'] * data['coll_primary_VV_offset']
data['vc_derived_rollover_risk'] = 0
data['vc_veh_type_SUV'] = 0
data['vc_veh_type_CAR'] = 0
data['vc_veh_type_VAN'] = 0
data['vc_veh_type_TRUCK'] = 0

In [ ]:
# Drop columns from config
drop_cols = cfg['columns']['drop_columns']
cols_to_drop = [c for c in drop_cols if c in data.columns]
data = data.drop(cols_to_drop, axis=1)
print(f"Dropped {len(cols_to_drop)} columns")

In [ ]:
# Define column sets
pass_through_cols = cfg['columns']['pass_through_columns']
all_feature_cols = [c for c in data.columns if c.startswith('vc_')]

# Keep only selected features + pass-through columns (match original behavior)
# This REMOVES non-selected columns entirely instead of zeroing them
cols_to_keep = pass_through_cols + selected_features
cols_to_keep = [c for c in cols_to_keep if c in data.columns]
data = data[cols_to_keep]

print(f"Kept {len(cols_to_keep)} columns (removed {len(all_feature_cols) - len(selected_features)} unselected features)")

In [ ]:
# Load monotonicity tables
trim_eda_table(support_path)
options_mono_table(support_path)

# Blank out trim fields
for idx, field in enumerate(trim_blank_list):
    if field in data.columns:
        if trim_blank_dtyp[idx] == 'bool':
            data[field] = False
        else:
            data[field] = data[field].mode().values[0]

# Blank out options fields
for field in options_blank_list:
    if field in data.columns:
        data[field] = False

In [ ]:
# Apply clipping from config
if clipping_df is not None:
    for _, row in clipping_df.iterrows():
        if row['feature_name'] in data.columns:
            box_var(data, row['feature_name'], row['min'], row['max'])

In [ ]:
# Apply trim clipping from monotonicity settings
trim_eda_include = trim_eda.loc[trim_eda['mono'] != 'x'].copy()
trim_clip_settings = trim_eda_include.loc[trim_eda_include['low_clip'].isna() == False][['field', 'low_clip', 'high_clip']].copy()

for _, row in trim_clip_settings.iterrows():
    if row['field'] in data.columns:
        box_var(data, row['field'], row['low_clip'], row['high_clip'])

In [ ]:
# Fix bool/float issues
dtype_df = data.dtypes.reset_index().rename(columns={'index': 'field', 0: 'dtype'})
bool_cols = dtype_df.loc[dtype_df['dtype'] == 'bool']['field'].tolist()
if 'train' in bool_cols:
    bool_cols.remove('train')

for col in bool_cols:
    data[col] = np.where(data[col] == True, 1, 0).astype('uint8')

# Round float precision columns
float_cols = cfg['columns'].get('float_precision_columns', [])
for col in float_cols:
    if col in data.columns:
        data[col] = round(data[col], 4)

In [ ]:
# Blank out features from feature_selection.csv
non_bools = trim_eda.loc[trim_eda['dtype'] != 'bool']['field'].tolist()

for field in blank_features:
    if field in data.columns:
        if field in non_bools:
            data[field] = data[field].mode().values[0]
        else:
            data[field] = 0

# Also blank out features from additional_blank_list in config
additional_blank_list = cfg.get('additional_blank_list', [])
for field in additional_blank_list:
    if field in data.columns:
        if field in non_bools:
            data[field] = data[field].mode().values[0]
        else:
            data[field] = 0

print(f"Blanked out {len(blank_features)} features from feature_selection")
print(f"Blanked out {len(additional_blank_list)} features from additional_blank_list")

In [ ]:
# Prepare model columns
facts = [c for c in data.columns if '_liab' in c or '_coll' in c or '_comp' in c]
features = [c for c in data.columns if c not in ['index', 'train', 'vc_Vehicle_Type_M', 'offset_AME_VT_VV'] and c not in facts]

data['weight'] = data['ee_' + cov]
data['num'] = data['incurred_cap_AME_VV_VT_' + cov]

# Set denom and offset from config
denom_col = cfg['columns']['denom_column']
offset_col = cfg['columns']['offset_column']
data['denom'] = data[denom_col]
data['offset'] = data[offset_col]

print(f"Features: {len(features)}")
print(f"Denom column: {denom_col}")
print(f"Offset column: {offset_col}")

In [ ]:
# Get monotonicity list
get_mono_list()

In [ ]:
# XGBoost parameters from config
xgb_cfg = cfg['xgboost']
tweedie_p = xgb_cfg['tweedie_p']
bmf = xgb_cfg['base_margin_factor']
max_depth = xgb_cfg['max_depth']
num_round = xgb_cfg['num_round']
min_child_weight = xgb_cfg['min_child_weight']
subsample = xgb_cfg['subsample']
colsample_bytree = xgb_cfg['colsample_bytree']
alph = xgb_cfg['alpha']

In [ ]:
# Train/test split and model training
data_train = data.loc[data['train'] == True].copy()
data_test = data.loc[data['train'] == False].copy()

print(f"Train: {data_train.shape}")
print(f"Test: {data_test.shape}")

xbg_wrapper(data_train, data_test, 'VIN_Date', 'num', 'denom', 'weight', 'offset', mono_list, tweedie_p, bmf, max_depth, num_round, min_child_weight, subsample, colsample_bytree, alph)

In [ ]:
# Lift charts
print("Test Lift Chart:")
lift_chart(out_test, 'weight', 10, print_table=True)

In [ ]:
print("Train Lift Chart:")
lift_chart(out_train, 'weight', 10, print_table=True)

In [ ]:
lift_chart_2026update(out_train, 'weight', 10, print_table=True)

In [ ]:
lift_chart_2026update(out_test, 'weight', 10, print_table=True)

In [ ]:
# Merge with EP data
cols_to_use = [c for c in data_ep.columns if c not in out_train.columns or c == 'VIN_Date']
out_train2 = out_train.merge(data_ep[cols_to_use], on='VIN_Date', how='left')
print(f"Merged shape: {out_train2.shape}")

In [ ]:
lift_chart_2026update_earned_premium(out_train2, 'weight', 10, print_table=True, earned_premium_col='ep_coll', relativity_base='pred')

In [ ]:
# SHAP analysis (using original working pattern)
all_feature_names = model_xgb.feature_names
train_sample = data_train[all_feature_names].sample(n=int(data_train.shape[0] * 0.20), random_state=42)

explainer = shap.Explainer(model_xgb)
dtrain = xgb.DMatrix(train_sample, feature_names=model_xgb.feature_names)
shap_values = explainer(dtrain)
shap_values.feature_names = all_feature_names

shap.plots.beeswarm(shap_values, max_display=40)

In [ ]:
shap.plots.bar(shap_values, max_display=40)

In [ ]:
# SHAP importance percentage
importance = np.abs(shap_values.values).mean(axis=0)
pct_importance = importance / importance.sum() * 100

imp_df = pd.DataFrame({
    'feature': shap_values.feature_names,
    'importance': importance,
    'pct_importance': pct_importance
})
imp_df = imp_df.sort_values('pct_importance', ascending=False).head(20)
display(imp_df)

In [ ]:
# Save model
output_dir = config_dir.replace('config/', 'output/', 1)
os.makedirs(output_dir, exist_ok=True)

if not DEBUG:
    model_path = os.path.join(output_dir, f"{veh_type}_{cov}_model.json")
    model_xgb.save_model(model_path)
    print(f"Model saved: {model_path}")
else:
    print("Debug mode - model not saved")

In [ ]:
print(f"\nExperiment complete: {cfg['experiment']['name']}")

In [ ]:
# Memory cleanup - important when running via papermill
# Delete large objects and force garbage collection
import gc

# Delete large dataframes
large_vars = ['data_ini', 'data', 'data_train', 'data_test', 'data_ep',
              'out_train', 'out_test', 'out_train2', 'train_sample',
              'shap_values', 'explainer', 'dtrain', 'model_xgb']

for var in large_vars:
    if var in dir():
        exec(f'del {var}')

gc.collect()
print("Memory cleanup complete")